# RL dispatch lab: tabular Q-learning for urgent orders (owner: Harsh, Slide 7 / Q3)

**Illustrative, small simulation.** DubaiDash Express is fictional, every cost is *assumed* (`data_seed/assumptions.json`), every result is *simulated*. This notebook is the "next step" on Slides 3 and 7, not a claim in the business case: nothing here enters `docs/frozen_results.json`.

**Question.** When an urgent order (60-minute promise) arrives, should dispatch send a van, send a motorbike, or wait 10 minutes for a better vehicle? Can a learned policy beat simple rules?

**Set-up (MDP, written out for the viva).**
- *Episode* = one urgent order, from arrival until it is assigned (a short, undiscounted episode). Days are sequences of orders that share a small urgent pool (2 vans + 3 motorbikes, assumed), so one decision changes the state seen by the next order.
- *Decision points* = order arrival, and every 10 minutes after a "defer" (a semi-MDP: time between decisions is uneven).
- *State* (4 x 3 x 3 x 3 = 108 buckets, of which 72 are reachable: orders arrive with 60 minutes and wait at most 3 x 10, so slack never drops below 30): time of day (08-11, 11-14, 14-17, 17-20), slack to the promise (< 25, 25-45, > 45 min), motorbike availability (free now, free within 10 min, not within 10 min), distance from the hub (< 6, 6-12, > 12 km).
- *Actions*: van, bike, defer. A fixed rule picks the actual vehicle: the one of that type that reaches the customer first (it returns to the hub for the parcel, then drives to the customer; idle vehicles wait at their last drop). Waiting is allowed at most 3 times per order (a 15-minute slack mask exists in the code but never binds).
- *Reward* = -(km x AED/km + driver minutes x 28/60 AED + 15 AED if late), booked when the order is assigned. A defer earns 0 and inherits the value of the same order's next decision.
- *Partially observable*: 5 vehicle positions and busy times and future arrivals are squeezed into 4 buckets, so the agent learns the best policy *over a coarse summary*; convergence to the true optimum is not guaranteed under state aggregation.

**Fair test.** Baselines see exactly the same days (common random numbers: each day depends only on its seed, never on the policy). Train days 1-3,000, validation days 5,001-5,030 (learning curve; 5,001-5,100 used once to choose alpha), test days 10,001-10,200 (held out, used once). Five training runs with different exploration seeds; we report the **median** run, plus a paired bootstrap 95 % interval on the test days.
- *Greedy*: nearest vehicle (any type that reaches the customer first). Never waits.
- *Cost-aware rule*: the type with the lowest immediate cost including the SLA penalty. Never waits. This is the real bar: beating greedy alone could just mean "bikes are cheaper per km".

## 1. Inputs (assumed) and design choices of this simulation

In [1]:
import json, math
from pathlib import Path
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
A = json.loads((ROOT / "data_seed" / "assumptions.json").read_text(encoding="utf-8"))

# Every input below is ASSUMED (assumptions.json) or a design choice of this small simulation (labelled).
VAN, BIKE = A["vehicles"]["van"], A["vehicles"]["bike"]
COST_KM = {"van": VAN["cost_per_km_aed"], "bike": BIKE["cost_per_km_aed"]}
SERVICE = {"van": VAN["service_time_min"], "bike": BIKE["service_time_min"]}
DRIVER_PER_MIN = A["costs"]["driver_cost_per_hour_aed"] / 60
SLA_PENALTY = A["costs"]["sla_penalty_aed"]
WINDOW = A["orders"]["urgent_window_min"]                     # urgent orders: 60-minute promise
DETOUR = A["fallback"]["haversine_detour_factor"]             # straight line -> road km
SPEED = A["fallback"]["average_speed_kmh_free_flow"] / 60     # km per minute, free flow
BIKE_SENS = A["fallback"]["bike_traffic_sensitivity"]         # bikes feel 60 % of a slow-down
TRAFFIC = {int(h): m for h, m in A["traffic_hour_multipliers"].items()}
URGENT_PER_DAY = round(A["orders"]["count"] * A["orders"]["urgent_share"])   # 300 x 0.10 = 30
SURGE_ORDERS = A["orders"]["urgent_surge_orders"]                            # 25

# Design choices of this simulation (not in assumptions.json; labelled "illustrative" on the page)
POOL = {"van": 2, "bike": 3}      # dedicated urgent-response vehicles
RADIUS_KM = 15                    # customers within 15 km (straight line) of the hub
SURGE_DAY_SHARE = 0.25            # one day in four has a 25-order surge inside one hour
DEFER_MIN = 10                    # "defer" waits 10 minutes, then decides again
MAX_DEFERS = 3
FORCE_SLACK = A["sla"]["risk_buffer_min"]   # below 15 min slack: no more waiting
DAY_START, DAY_END = 8 * 60 + 30, 19 * 60

def traffic(minute, vtype):
    m = TRAFFIC.get(min(19, max(8, int(minute // 60))), 1.0)
    return 1 + BIKE_SENS * (m - 1) if vtype == "bike" else m

def road_km(p, q):
    return DETOUR * math.hypot(p[0] - q[0], p[1] - q[1])

## 2. Simulated days (seeded; train, validation and test seeds never overlap)
One day = 30 urgent orders (300 orders x 10 % urgent share, assumed) between 08:30 and 19:00; one day in four adds the assumed 25-order urgent surge inside one hour. Customers are within 15 km of the hub (straight line x 1.35 detour).

In [2]:
def make_day(seed):
    """One simulated day of urgent orders. Depends only on the day seed (common random numbers for every policy)."""
    rng = np.random.default_rng(seed)
    n = URGENT_PER_DAY
    times = list(rng.uniform(DAY_START, DAY_END - WINDOW, n))
    if rng.random() < SURGE_DAY_SHARE:
        start = rng.uniform(11 * 60, 17 * 60)
        times += list(start + rng.uniform(0, 60, SURGE_ORDERS))
    times = np.sort(np.array(times))
    r = RADIUS_KM * np.sqrt(rng.random(len(times)))
    th = rng.uniform(0, 2 * np.pi, len(times))
    return [{"id": i, "t": float(t), "xy": (float(x), float(y)), "due": float(t + WINDOW), "surge_day": len(times) > n}
            for i, (t, x, y) in enumerate(zip(times, r * np.cos(th), r * np.sin(th)))]

TRAIN_SEEDS = range(1, 3001)            # 3,000 training days
VAL_SEEDS = range(5001, 5031)           # 30 fixed validation days (learning curve)
TEST_SEEDS = range(10001, 10201)        # 200 held-out test days, never seen in training or tuning
assert not set(TRAIN_SEEDS) & set(VAL_SEEDS) and not set(TRAIN_SEEDS) & set(TEST_SEEDS) and not set(VAL_SEEDS) & set(TEST_SEEDS)

## 3. Environment: fleet, state and one simulated day

In [3]:
HUB = (0.0, 0.0)

class Fleet:
    def __init__(self):
        self.v = [{"type": t, "free": DAY_START - 30, "xy": HUB} for t in ("van", "bike") for _ in range(POOL[t])]

    def option(self, vtype, order, now):
        """Earliest-arrival vehicle of a type: it returns to the hub for the parcel, then drives to the customer."""
        best = None
        for k, v in enumerate(self.v):
            if v["type"] != vtype:
                continue
            leave = max(now, v["free"])
            km1, km2 = road_km(v["xy"], HUB), road_km(HUB, order["xy"])
            t_hub = leave + km1 / SPEED * traffic(leave, vtype)
            arrive = t_hub + km2 / SPEED * traffic(t_hub, vtype)
            done = arrive + SERVICE[vtype]
            late = arrive > order["due"]
            cost = (km1 + km2) * COST_KM[vtype] + (done - leave) * DRIVER_PER_MIN + (SLA_PENALTY if late else 0)
            o = {"k": k, "type": vtype, "arrive": arrive, "done": done, "late": late, "km": km1 + km2, "cost": cost,
                 "km_cost": (km1 + km2) * COST_KM[vtype], "drv_cost": (done - leave) * DRIVER_PER_MIN}
            if best is None or o["arrive"] < best["arrive"]:
                best = o
        return best

    def commit(self, o, order):
        v = self.v[o["k"]]
        v["free"], v["xy"] = o["done"], order["xy"]

    def bike_bucket(self, now):
        soon = min(v["free"] for v in self.v if v["type"] == "bike") - now
        return 0 if soon <= 0 else 1 if soon <= DEFER_MIN else 2

# State = (time bucket, slack bucket, bike availability, distance bucket): 4 x 3 x 3 x 3 = 108 states
def state(order, now, fleet):
    tb = 0 if now < 11 * 60 else 1 if now < 14 * 60 else 2 if now < 17 * 60 else 3
    slack = order["due"] - now
    sb = 0 if slack < 25 else 1 if slack < 45 else 2
    d = road_km(HUB, order["xy"])
    db = 0 if d < 6 else 1 if d < 12 else 2
    return ((tb * 3 + sb) * 3 + fleet.bike_bucket(now)) * 3 + db

N_STATES, ACTIONS = 108, ("van", "bike", "defer")
# Reachable: orders arrive with 60 min of slack and may wait at most 3 x 10 min, so slack never drops below 30 min:
# the "< 25 min" bucket (36 states) is never visited and the 15-minute defer mask never binds. 72 states are reachable.
REACHABLE_STATES = 72

def allowed_actions(order, now, defers):
    """Defer is not allowed below 15 min slack or after 3 defers."""
    return (0, 1) if order["due"] - now < FORCE_SLACK or defers >= MAX_DEFERS else (0, 1, 2)

def run_day(day, policy, learn=None):
    """Plays one day. Decisions happen when an urgent order arrives or a deferred order comes back.
    policy(s, order, now, fleet, allowed) -> action. learn(s, a, r, s_next, allowed_next) for Q-learning, where
    s_next = None means the order is assigned (its short episode ends). Each order is its own episode:
    a defer is charged the full cost of that same order's later assignment (no discounting inside an order)."""
    fleet = Fleet()
    queue = sorted((o["t"], o["id"], 0, None) for o in day)   # (decision time, order id, defers so far, previous (s, a))
    out = {"orders": len(day), "cost": 0.0, "late": 0, "van": 0, "defer": 0, "km": 0.0, "km_cost": 0.0, "drv_cost": 0.0, "sla_cost": 0.0,
           "surge": int(bool(day) and day[0]["surge_day"])}
    while queue:
        now, oid, defers, prev = queue.pop(0)
        order = day[oid]
        s = state(order, now, fleet)
        allowed = allowed_actions(order, now, defers)
        if prev is not None and learn:
            learn(prev[0], prev[1], 0.0, s, allowed)
        a = policy(s, order, now, fleet, allowed)
        if a == 2:
            out["defer"] += 1
            queue.append((now + DEFER_MIN, oid, defers + 1, (s, a)))
            queue.sort(key=lambda q: (q[0], q[1]))
            continue
        o = fleet.option(ACTIONS[a], order, now)
        fleet.commit(o, order)
        out["cost"] += o["cost"]; out["km"] += o["km"]; out["km_cost"] += o["km_cost"]; out["drv_cost"] += o["drv_cost"]
        out["late"] += o["late"]; out["sla_cost"] += SLA_PENALTY * o["late"]; out["van"] += o["type"] == "van"
        if learn:
            learn(s, a, -o["cost"], None, ())
    return out

## 4. Baselines

In [4]:
def greedy(s, order, now, fleet, allowed):
    """Nearest vehicle: the vehicle (any type) that reaches the customer first. Never waits."""
    return min((0, 1), key=lambda a: fleet.option(ACTIONS[a], order, now)["arrive"])

def myopic(s, order, now, fleet, allowed):
    """Cost-aware rule: the type with the lowest immediate cost including the SLA penalty if it would be late. Never waits."""
    return min((0, 1), key=lambda a: fleet.option(ACTIONS[a], order, now)["cost"])

def make_random(seed):
    rng = np.random.default_rng(seed)
    return lambda s, order, now, fleet, allowed: int(rng.choice(allowed))

## 5. Tabular Q-learning
Epsilon-greedy exploration from 1.0 down to 0.05 over 70 % of the training days; Q starts at 0, which is optimistic because every reward is a cost (it pushes exploration). The learning curve is measured with exploration switched off (epsilon = 0) on the fixed validation days.

In [5]:
ALPHA, EPS_START, EPS_END = 0.05, 1.0, 0.05   # alpha and 3,000 days chosen on validation days only (5001-5100), never on test days

def train(agent_seed, train_seeds=TRAIN_SEEDS, val_every=100, val_seeds=VAL_SEEDS):
    rng = np.random.default_rng(10_000_000 + agent_seed)   # exploration RNG, separate from the day RNG
    Q = np.zeros((N_STATES, 3))                            # Q = 0 with negative rewards = optimistic start
    visits = np.zeros((N_STATES, 3), dtype=int)
    n = len(train_seeds)
    curve = []

    def learn(s, a, r, s2, allowed2):
        visits[s, a] += 1
        target = r if s2 is None else r + max(Q[s2, b] for b in allowed2)
        Q[s, a] += ALPHA * (target - Q[s, a])

    def eps_policy(eps):
        def pol(s, order, now, fleet, allowed):
            if rng.random() < eps:
                return int(rng.choice(allowed))
            return max(allowed, key=lambda b: Q[s, b])
        return pol

    for ep, day_seed in enumerate(train_seeds, start=1):
        eps = max(EPS_END, EPS_START - (EPS_START - EPS_END) * ep / (0.7 * n))
        run_day(make_day(day_seed), eps_policy(eps), learn)
        if ep % val_every == 0 or ep == 1:
            curve.append((ep, evaluate(greedy_q(Q), val_seeds)["cost_per_order"]))
    return Q, visits, curve

def greedy_q(Q):
    return lambda s, order, now, fleet, allowed: max(allowed, key=lambda b: Q[s, b])

def evaluate(policy, seeds, per_day=False):
    days = [run_day(make_day(sd), policy) for sd in seeds]
    tot = {k: sum(d[k] for d in days) for k in days[0] if k != "surge"}
    def on_time(sel):
        ds = [d for d in days if sel(d)]
        return 100 * (1 - sum(d["late"] for d in ds) / max(1, sum(d["orders"] for d in ds)))
    res = {
        "cost_per_order": tot["cost"] / tot["orders"],
        "on_time_pct": 100 * (1 - tot["late"] / tot["orders"]),
        "van_share_pct": 100 * tot["van"] / tot["orders"],
        "defers_per_order": tot["defer"] / tot["orders"],
        "km_per_order": tot["km"] / tot["orders"],
        "split_per_order": {k: tot[k] / tot["orders"] for k in ("km_cost", "drv_cost", "sla_cost")},
        "on_time_normal_pct": on_time(lambda d: not d["surge"]),
        "on_time_surge_pct": on_time(lambda d: d["surge"]),
        "surge_days": sum(d["surge"] for d in days),
    }
    if per_day:
        res["per_day"] = [(d["cost"], d["orders"]) for d in days]   # totals, for a ratio-of-sums bootstrap
    return res

## 6. Train five runs

In [6]:
AGENT_SEEDS = [0, 1, 2, 3, 4]          # five independent training runs (exploration RNG only; days are shared)
runs = []
for sd in AGENT_SEEDS:
    Q, visits, curve = train(sd)
    runs.append({"seed": sd, "Q": Q, "visits": visits, "curve": curve})
    print(f"agent seed {sd}: validation cost per urgent order {curve[0][1]:.2f} -> {curve[-1][1]:.2f} AED")

greedy_val = evaluate(greedy, VAL_SEEDS)["cost_per_order"]
myopic_val = evaluate(myopic, VAL_SEEDS)["cost_per_order"]
print(f"validation reference: greedy {greedy_val:.2f}, cost-aware rule {myopic_val:.2f} AED per urgent order")

agent seed 0: validation cost per urgent order 57.13 -> 35.53 AED


agent seed 1: validation cost per urgent order 57.15 -> 35.61 AED


agent seed 2: validation cost per urgent order 48.17 -> 35.81 AED


agent seed 3: validation cost per urgent order 53.25 -> 36.01 AED


agent seed 4: validation cost per urgent order 49.51 -> 36.29 AED
validation reference: greedy 37.83, cost-aware rule 35.21 AED per urgent order


## 7. Held-out test (200 days)

In [7]:
N_BOOT = 2000
BOOT_IDX = np.random.default_rng(7).integers(0, len(TEST_SEEDS), (N_BOOT, len(TEST_SEEDS)))   # same resampled days for every policy (paired)

def ratio(per_day, idx):
    """Cost per urgent order = sum of cost / sum of orders over the (resampled) days, the same pooled figure as the headline."""
    c, o = np.asarray(per_day, dtype=float).T
    return c[idx].sum(axis=-1) / o[idx].sum(axis=-1)

def boot_ci(per_day):
    r = ratio(per_day, BOOT_IDX)
    return float(np.percentile(r, 2.5)), float(np.percentile(r, 97.5))

test = {"greedy": evaluate(greedy, TEST_SEEDS, per_day=True),
        "myopic": evaluate(myopic, TEST_SEEDS, per_day=True),
        "random": evaluate(make_random(99), TEST_SEEDS, per_day=True)}
q_tests = [evaluate(greedy_q(r["Q"]), TEST_SEEDS, per_day=True) for r in runs]

def no_wait(Q):
    """Ablation: the same Q-table with waiting switched off (van or bike only)."""
    return lambda s, order, now, fleet, allowed: max((0, 1), key=lambda b: Q[s, b])
q_costs = [t["cost_per_order"] for t in q_tests]
median_run = int(np.argsort(q_costs)[len(q_costs) // 2])   # report the median of the five runs, not the best
test["q_learning"] = q_tests[median_run]
test["q_no_wait"] = evaluate(no_wait(runs[median_run]["Q"]), TEST_SEEDS, per_day=True)

def paired(a, b):
    """Difference in pooled cost per order (a - b, negative = a cheaper), bootstrapped over the same resampled days."""
    pa, pb = test[a]["per_day"], test[b]["per_day"]
    diff = ratio(pa, BOOT_IDX) - ratio(pb, BOOT_IDX)
    point = test[a]["cost_per_order"] - test[b]["cost_per_order"]
    day_a = np.array([c / o for c, o in pa]); day_b = np.array([c / o for c, o in pb])
    return {"mean_diff_aed": float(point), "ci95": [float(np.percentile(diff, 2.5)), float(np.percentile(diff, 97.5))],
            "days_cheaper_pct": float(100 * (day_a < day_b).mean())}

paired_vs = {"greedy": paired("q_learning", "greedy"), "myopic": paired("q_learning", "myopic")}
for name, t in test.items():
    lo, hi = boot_ci(t["per_day"])
    t["ci95"] = [lo, hi]
    print(f"{name:11s} {t['cost_per_order']:6.2f} AED/order (95% CI {lo:.2f}-{hi:.2f})  on-time {t['on_time_pct']:5.1f}% (normal days {t['on_time_normal_pct']:.1f}%, surge days {t['on_time_surge_pct']:.1f}%)  van share {t['van_share_pct']:5.1f}%  waits/order {t['defers_per_order']:.2f}")
print(f"surge days among the {len(TEST_SEEDS)} test days: {test['greedy']['surge_days']}")
print(f"waiting is worth {test['q_no_wait']['cost_per_order'] - test['q_learning']['cost_per_order']:.2f} AED per order of the {test['greedy']['cost_per_order'] - test['q_learning']['cost_per_order']:.2f} AED saving vs greedy")
print("Q-learning runs on the test days:", [round(c, 2) for c in q_costs])
for b, p in paired_vs.items():
    print(f"Q-learning minus {b}: {p['mean_diff_aed']:+.2f} AED/order (95% CI {p['ci95'][0]:+.2f} to {p['ci95'][1]:+.2f}), cheaper on {p['days_cheaper_pct']:.0f}% of days")

greedy       38.26 AED/order (95% CI 37.38-39.16)  on-time  72.1% (normal days 90.3%, surge days 36.8%)  van share  36.5%  waits/order 0.00
myopic       35.45 AED/order (95% CI 34.69-36.19)  on-time  71.0% (normal days 85.3%, surge days 43.2%)  van share  18.4%  waits/order 0.00
random       42.61 AED/order (95% CI 41.86-43.36)  on-time  58.5% (normal days 71.3%, surge days 33.6%)  van share  50.1%  waits/order 0.48
q_learning   35.71 AED/order (95% CI 34.99-36.46)  on-time  67.5% (normal days 80.7%, surge days 42.0%)  van share  16.9%  waits/order 0.49
q_no_wait    35.84 AED/order (95% CI 35.10-36.60)  on-time  67.5% (normal days 80.9%, surge days 41.7%)  van share  18.1%  waits/order 0.00
surge days among the 200 test days: 44
waiting is worth 0.13 AED per order of the 2.55 AED saving vs greedy
Q-learning runs on the test days: [35.64, 35.64, 35.71, 35.79, 36.0]
Q-learning minus greedy: -2.55 AED/order (95% CI -2.82 to -2.28), cheaper on 88% of days
Q-learning minus myopic: +0.26 AED

## 8. What the policy learned (median run, indicative)
Share of decisions per action by slack and motorbike availability, weighted by how often each state was visited. States visited fewer than 30 times are left out as "not learned". Indicative only: in many states the three action values are within 1 to 3 AED of each other (near-ties).

In [8]:
TIME_LABELS = ["08-11", "11-14", "14-17", "17-20"]
SLACK_LABELS = ["< 25 min", "25-45 min", "> 45 min"]
BIKE_LABELS = ["bike free now", "bike free within 10 min", "no bike within 10 min"]
DIST_LABELS = ["< 6 km", "6-12 km", "> 12 km"]
Qm, Vm = runs[median_run]["Q"], runs[median_run]["visits"]
policy_rows = []
for sb in range(3):
    for bb in range(3):
        counts = np.zeros(3)
        n = 0
        for tb in range(4):
            for db in range(3):
                s = ((tb * 3 + sb) * 3 + bb) * 3 + db
                v = int(Vm[s].sum())
                if v < 30:
                    continue  # too few visits: "not learned"
                allowed = (0, 1) if sb == 0 else (0, 1, 2)
                counts[max(allowed, key=lambda b: Qm[s, b])] += v
                n += v
        if n == 0:
            continue  # never reached: orders arrive with 60 min of slack and may wait at most 3 x 10 min
        share = (counts / n * 100).round(1).tolist()
        policy_rows.append({"slack": SLACK_LABELS[sb], "bike": BIKE_LABELS[bb], "visits": n,
                            "share_pct": dict(zip(ACTIONS, share))})
for r in policy_rows:
    print(f"{r['slack']:10s} | {r['bike']:24s} | visits {r['visits']:6d} | {r['share_pct']}")

25-45 min  | bike free now            | visits   7387 | {'van': 21.4, 'bike': 14.4, 'defer': 64.2}
25-45 min  | bike free within 10 min  | visits   2469 | {'van': 23.6, 'bike': 50.5, 'defer': 25.9}
25-45 min  | no bike within 10 min    | visits  20128 | {'van': 3.5, 'bike': 23.1, 'defer': 73.4}
> 45 min   | bike free now            | visits  76276 | {'van': 6.0, 'bike': 94.0, 'defer': 0.0}
> 45 min   | bike free within 10 min  | visits  11411 | {'van': 2.8, 'bike': 74.7, 'defer': 22.5}
> 45 min   | no bike within 10 min    | visits  57649 | {'van': 27.4, 'bike': 21.2, 'defer': 51.4}


## 9. Export for /rl-lab

In [9]:
r2 = lambda x: round(float(x), 2)
def summary(t):
    return {"cost_per_order": r2(t["cost_per_order"]), "ci95": [r2(v) for v in t["ci95"]], "on_time_pct": round(t["on_time_pct"], 1),
            "van_share_pct": round(t["van_share_pct"], 1), "defers_per_order": r2(t["defers_per_order"]), "km_per_order": r2(t["km_per_order"]),
            "split_per_order": {k: r2(v) for k, v in t["split_per_order"].items()},
            "on_time_normal_pct": round(t["on_time_normal_pct"], 1), "on_time_surge_pct": round(t["on_time_surge_pct"], 1)}

eps = [c[0] for c in runs[0]["curve"]]
curve_rows = []
for i, ep in enumerate(eps):
    vals = [r["curve"][i][1] for r in runs]
    curve_rows.append({"episode": ep, "q_median": r2(np.median(vals)), "q_min": r2(min(vals)), "q_max": r2(max(vals))})

beats_greedy = paired_vs["greedy"]["ci95"][1] < 0
beats_myopic = paired_vs["myopic"]["ci95"][1] < 0
worse_myopic = paired_vs["myopic"]["ci95"][0] > 0
verdict_code = "beats_both" if beats_greedy and beats_myopic else "beats_greedy_only" if beats_greedy else "no_better"
qd = test["q_learning"]
verdict = (
    f"On {len(TEST_SEEDS)} held-out simulated days, Q-learning costs {qd['cost_per_order']:.2f} AED per urgent order vs "
    f"{test['greedy']['cost_per_order']:.2f} for nearest-vehicle greedy ({'cheaper' if beats_greedy else 'not clearly cheaper'}, "
    f"95% CI of the difference {paired_vs['greedy']['ci95'][0]:+.2f} to {paired_vs['greedy']['ci95'][1]:+.2f}) and "
    f"{test['myopic']['cost_per_order']:.2f} for a simple cost-aware rule ("
    + ("cheaper" if beats_myopic else "more expensive" if worse_myopic else "no clear difference")
    + f", CI {paired_vs['myopic']['ci95'][0]:+.2f} to {paired_vs['myopic']['ci95'][1]:+.2f}). "
    + ("It beats both baselines." if beats_myopic and beats_greedy else
       "It learns roughly the cost-aware rule (send a motorbike unless it would be late) but cannot beat it: with per-order rewards it optimises the same cost as the rule with less information (4 buckets instead of exact arrival times). The knock-on effect of one dispatch on later orders is not modelled yet." if beats_greedy else
       "It does not beat the baselines in this small simulation.")
)
print(verdict)

out = {
    "_comment": "Illustrative, small simulation. Written by notebook/rl_dispatch.ipynb (Restart & Run All). Fictional company, assumed costs. Not part of docs/frozen_results.json and not quoted on any slide as a result.",
    "generated_by": "notebook/rl_dispatch.ipynb",
    "assumptions_version": A["version"],
    "setup": {
        "pool": POOL, "urgent_orders_per_day": URGENT_PER_DAY, "surge_day_share": SURGE_DAY_SHARE, "surge_orders": SURGE_ORDERS,
        "window_min": WINDOW, "defer_min": DEFER_MIN, "max_defers": MAX_DEFERS, "force_slack_min": FORCE_SLACK, "radius_km": RADIUS_KM,
        "states": N_STATES, "reachable_states": REACHABLE_STATES, "actions": list(ACTIONS), "alpha": ALPHA, "epsilon": [EPS_START, EPS_END],
        "train_days": [min(TRAIN_SEEDS), max(TRAIN_SEEDS)], "validation_days": [min(VAL_SEEDS), max(VAL_SEEDS)],
        "test_days": [min(TEST_SEEDS), max(TEST_SEEDS)], "agent_seeds": AGENT_SEEDS,
        "costs": {"van_aed_per_km": COST_KM["van"], "bike_aed_per_km": COST_KM["bike"], "driver_aed_per_hour": A["costs"]["driver_cost_per_hour_aed"], "sla_penalty_aed": SLA_PENALTY},
    },
    "learning_curve": curve_rows,
    "reference": {"greedy_validation": r2(greedy_val), "myopic_validation": r2(myopic_val)},
    "test": {name: summary(t) for name, t in test.items()},
    "q_learning_runs_test": [r2(c) for c in q_costs],
    "paired": {k: {"mean_diff_aed": r2(v["mean_diff_aed"]), "ci95": [r2(x) for x in v["ci95"]], "days_cheaper_pct": round(v["days_cheaper_pct"], 1)} for k, v in paired_vs.items()},
    "policy": policy_rows,
    "policy_note": "indicative: in many states the action values are within 1 to 3 AED of each other (near-ties)",
    "waiting_worth_aed": r2(test["q_no_wait"]["cost_per_order"] - test["q_learning"]["cost_per_order"]),
    "surge_test_days": test["greedy"]["surge_days"],
    "verdict_code": verdict_code,
    "verdict": verdict,
}
OUT = ROOT / "web" / "public" / "data" / "rl.json"
OUT.write_text(json.dumps(out, indent=2) + "\n", encoding="utf-8")
print("wrote", OUT.relative_to(ROOT))

On 200 held-out simulated days, Q-learning costs 35.71 AED per urgent order vs 38.26 for nearest-vehicle greedy (cheaper, 95% CI of the difference -2.82 to -2.28) and 35.45 for a simple cost-aware rule (more expensive, CI +0.11 to +0.41). It learns roughly the cost-aware rule (send a motorbike unless it would be late) but cannot beat it: with per-order rewards it optimises the same cost as the rule with less information (4 buckets instead of exact arrival times). The knock-on effect of one dispatch on later orders is not modelled yet.
wrote web/public/data/rl.json


## 10. Reading the result honestly
- The verdict printed above is generated from the numbers, not written by hand.
- With the assumed 15 AED SLA penalty, a late urgent order is cheap compared with a van trip, so any cost-minimising policy (learned or rule-based) accepts some lateness. A higher penalty would change the answer; a real pilot must price lateness from customer data. On surge days every policy misses many 60-minute promises: some edge-of-area orders cannot be reached in 60 minutes by a vehicle that must first return to the hub.
- The learner beats nearest-vehicle mainly by using motorbikes instead of vans; waiting adds only a little (the "waiting off" ablation above).
- With per-order rewards it optimises the same cost as the cost-aware rule but sees only 4 buckets instead of exact arrival times, so matching the rule is its ceiling. Learning the knock-on effect of one dispatch on later orders (a fleet-level reward) is the real next step, and it needs logged dispatch decisions from the pilot (shadow mode first). That is why RL is the **next step** after the 90-day pilot, not a claim today.